In [15]:
import bs4
import requests

URL = 'https://www.rcsb.org/downloads/fasta'

In [16]:
# First, fetch the entire sequence data in RCSB PDB
response = requests.get(URL)
soup = bs4.BeautifulSoup(response.text, 'html.parser')
link_to_download = [i for i in soup.find_all("a") if i.has_attr('href') and 'Download a file containing sequences in FASTA format for all entries in the PDB archive' in i.text][0]

In [17]:
# Download and extract the file now
from urllib.request import urlretrieve
urlretrieve(link_to_download['href'], 'pdb_seqres.txt.gz')

import gzip
import shutil
with gzip.open("pdb_seqres.txt.gz", "rb") as f_in, open("sequences.txt", "wb") as f_out:
    shutil.copyfileobj(f_in, f_out)

In [18]:
with open('sequences.txt', 'r') as f:
    sequences = f.readlines()

In [19]:
# It is ensured that each entry spans two lines:
#   - the first line holds the general information about the sequence
#   - the second line holds the sequence itself

print("Total entries in PDB:", len(sequences) // 2) # 1,049,262 sequences in total (not removing redundant sequences)

# There are multiple identifiers for all of the sequences, denoted using "mol:<sequence type>"
# We are interested in "mol:protein" entries only
# But remember that an entry spans two lines, so we have to iterate manually
proteins = []
for index in range(0, len(sequences), 2):
    if 'mol:protein' in sequences[index]:
        proteins.append(sequences[index].strip())
        proteins.append(sequences[index + 1].strip())

# Let's see how many protein sequences there are now
print("Total protein entries in PDB:", len(proteins) // 2) # 989,835
# A lot of them from the original set are proteins:
# 1,049,262 - 989,835 = 59,427 non-protein entries only
# Keep in mind that the protein sequence entries do repeat, just with different identifiers at the beginning

Total entries in PDB: 1049262
Total protein entries in PDB: 989835


In [20]:
# So we'll remove redundant sequences now
unique_sequences = []
for i in range(0, len(proteins), 2):
    # This format is as follows:
    # PDB identifier, length of sequence, protein name, protein sequence
    unique_sequences.append([proteins[i].split(' ')[0][1:], proteins[i].split(' ')[2].split(':')[1], ' '.join(proteins[i].split(' ')[4:]), proteins[i + 1]])

# We can see the format:
print(unique_sequences[15])

['109m_A', '154', 'MYOGLOBIN', 'MVLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRFKHLKTEAEMKASEDLKKHGVTVLTALGAILKKKGHHEAELKPLAQSHATKHKIPIKYLEFISEAIIHVLHSRHPGNFGADAQGAMNKALELFRKDIAAKYKELGYQG']


In [21]:
# Now we have to compare if the second, third, and fourth elements match with any other tuple.
# If so, take the PDB identifier at the first element and append it with the original entry.
# This will be a O(n^2) operation unfortunately.

# We won't run it because it will take a monumental time when executing serially
# For reference, it took 3 seconds to iterate over 989,835 entries
# n^2 means we'll wait for roughly 34 days to compute this
def extract_unique_entries():
    unique_entries = {}
    for i in range(len(unique_sequences)):
        pdb_ids = unique_sequences[i][0]
        entry = [unique_sequences[i][1], unique_sequences[i][2], unique_sequences[i][3]]
        for j in range(len(unique_sequences)):
            if (
                unique_sequences[i][1] == unique_sequences[j][1]
                and unique_sequences[i][2] == unique_sequences[j][2]
                and unique_sequences[i][3] == unique_sequences[j][3]
            ):
                pdb_ids += '/' + unique_sequences[j][0]
        unique_entries[pdb_ids] = entry
    return unique_entries

# We'll proceed with the entries as is

In [22]:
# Before that, we have to pick the top 5 most common PTMs in the clean PTM data
import pandas as pd
from pathlib import Path
from typing import List
from bs4 import BeautifulSoup
import zipfile

DBPTM_URL = 'https://biomics.lab.nycu.edu.tw/dbPTM'
DATA_DIR = Path('dbptm_data')

# Function to download dbPTM data and extract files (TSVs)
def download_and_extract() -> List[Path]:
    DATA_DIR.mkdir(exist_ok=True)
    zips = []
    resp = requests.get(f'{DBPTM_URL}/download.php', timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    for a in soup.find_all('a', href=True):
        href = a['href']
        if 'experiment' in href and href.endswith('.zip'):
            url = f"{DBPTM_URL}/{href}"
            fn = DATA_DIR / url.split('/')[-1]
            if not fn.exists():
                with requests.get(url, stream=True, timeout=60) as r:
                    r.raise_for_status()
                    with open(fn, 'wb') as f:
                        for chunk in r.iter_content(1<<14):
                            if chunk: f.write(chunk)
            zips.append(fn)

    # extract all of the files and append the TSV files
    tsvs = []
    for z in zips:
        with zipfile.ZipFile(z, 'r') as zf:
            zf.extractall(DATA_DIR)
            for n in zf.namelist():
                if '.' not in n:
                    tsvs.append(DATA_DIR / n)
    return tsvs

_ = download_and_extract()


In [23]:
import pandas as pd
from glob import glob

PTMs = {}
for filepath in glob("dbptm_data/*.zip"):
    df = pd.read_csv(filepath.split('.')[0], sep='\t')
    PTMs[filepath.split('\\')[-1].split('.')[0]] = len(df)

In [24]:
PTMs_sorted = dict(sorted(PTMs.items(), key=lambda item: item[1], reverse=True))
PTMs_sorted
# pick out top 5 PTMs
top_5_PTMs = list(PTMs_sorted.keys())[:5]

In [25]:
top_5_PTMs
# We got the top 5 PTMs, now for each PTM, the most common residues they occur on are:
# Phosphorylation: S, T, Y
# Ubiquitination: K
# Acetylation: K
# N-linked Glycosylation: N
# Succinylation: K
# SO let's set that up
PTM_residues = {
    'Phosphorylation': ['S', 'T', 'Y'],
    'Ubiquitination': ['K'],
    'Acetylation': ['K'],
    'N-linked Glycosylation': ['N'],
    'Succinylation': ['K']
}

In [26]:
import pymongo
# Load the positional frequency matrices
def load_tables_from_mongo(
    mongo_uri: str = "mongodb://localhost:27017",
    db_name: str = "ptmkb",
    coll_name: str = "tables",
) -> dict:
    client = pymongo.MongoClient(mongo_uri)
    client["ptmkb"]["tables"].create_index([("ptm", pymongo.ASCENDING)], unique=True)
    coll = client[db_name][coll_name]
    

    response = {}

    # Each document: { "ptm": "Acetylation", "data": { "freq": {...}, "log-e": {...} } }
    cursor = coll.find({}, {"_id": 0, "ptm": 1, "data.freq": 1, "data.log-e": 1})

    for doc in cursor:
        ptm = doc["ptm"]
        freq_map = doc.get("data", {}).get("freq", {}) or {}
        loge_map = doc.get("data", {}).get("log-e", {}) or {}

        # Merge AA keys from both maps
        all_aas = set(freq_map) | set(loge_map)
        response[ptm] = {
            aa: {
                "log-e": loge_map.get(aa, {}),
                "freq":   freq_map.get(aa, {}),
            }
            for aa in all_aas
        }

    return response

TABLES = load_tables_from_mongo()

In [27]:
# Copying functions from previous case study
def get_longest_centered_array(values: list[float | int | str], tol: float = 1e-5) -> list[float]:
    cleaned = [float(v) if v in ('-inf', 'inf') else v for v in values]
    
    numeric_values = [v for v in cleaned if not isinstance(v, str)]
    if not numeric_values:
        raise ValueError("No numeric values found.")
    
    closest_idx = min(range(len(cleaned)), key=lambda i: abs(cleaned[i]) if not isinstance(cleaned[i], str) else float('inf'))
    
    left, right = closest_idx, closest_idx
    while left > 0 and cleaned[left - 1] != '-inf':
        left -= 1
    while right < len(cleaned) - 1 and cleaned[right + 1] != '-inf':
        right += 1
    
    return cleaned[left:right + 1]


def additive_calculator(vector: list[float | int | str]) -> float:
    additive_score = 0.0
    vector = get_longest_centered_array(vector)
    if len(vector) < 13:
        additive_score = '-INF'
    else:
        for value in vector:
            if isinstance(value, float | int):
                additive_score += value
    return additive_score

def construct_subsequence(protein: str, site0: int, flank: int = 10) -> str:
    start = max(0, site0 - flank)
    end = min(len(protein), site0 + flank + 1)
    subseq = protein[start:end]
    if site0 < flank:
        subseq = ('-' * (flank - site0)) + subseq
    if site0 + flank >= len(protein):  # fix: >=
        subseq += '-' * ((site0 + flank + 1) - len(protein))
    return subseq

In [28]:
# We'll have a condition for N-linked Glycosylation PTM that the residue must be in the sequon N-X-S/T
# Now iterate over all unique sequences and compute additive scores for each PTM
processed_data = []
for i, entry in enumerate(unique_sequences):
    row = {}
    if i % 10000 == 0 and i != 0:
        print("Done with", i, "entries")
    pdb_id = entry[0]
    sequence = entry[3]
    row["id"] = pdb_id
    row['seq'] = sequence

    for ptm in top_5_PTMs:
        ptm_score = 0.0
        target_residues = PTM_residues[ptm]

        row[ptm] = [] # where we store all scores for that PTM
        
        for idx, residue in enumerate(sequence):
            if residue in target_residues:
                subsequence = construct_subsequence(sequence, idx)

                center_idx = len(subsequence) // 2
                char = subsequence[center_idx].upper()
                keys = [f"+{i}" if i > 0 else str(i)
                    for i in range(-center_idx, center_idx + 1)]
                table = TABLES.get(ptm, {}).get(residue, {}).get('log-e', {})
                vector = [
                    table.get(key, {}).get(subsequence[idx], float('-inf'))
                    for idx, key in enumerate(keys)
                ]

                # Special condition for N-linked Glycosylation
                check = True
                if ptm == 'N-linked Glycosylation':
                    if idx + 2 < len(sequence):
                        if sequence[idx + 1] == 'P' or sequence[idx + 2] not in ['S', 'T']:
                            check = False
                    else:
                        check = False

                if check:
                    ptm_score = additive_calculator(vector)
                    row[ptm].append({"Site":idx + 1, "LOGSUM": ptm_score})
    processed_data.append(row)
                    

Done with 10000 entries
Done with 20000 entries
Done with 30000 entries
Done with 40000 entries
Done with 50000 entries
Done with 60000 entries
Done with 70000 entries
Done with 80000 entries
Done with 90000 entries
Done with 100000 entries
Done with 110000 entries
Done with 120000 entries
Done with 130000 entries
Done with 140000 entries
Done with 150000 entries
Done with 160000 entries
Done with 170000 entries
Done with 180000 entries
Done with 190000 entries
Done with 200000 entries
Done with 210000 entries
Done with 220000 entries
Done with 230000 entries
Done with 240000 entries
Done with 250000 entries
Done with 260000 entries
Done with 270000 entries
Done with 280000 entries
Done with 290000 entries
Done with 300000 entries
Done with 310000 entries
Done with 320000 entries
Done with 330000 entries
Done with 340000 entries
Done with 350000 entries
Done with 360000 entries
Done with 370000 entries
Done with 380000 entries
Done with 390000 entries
Done with 400000 entries
Done with

In [ ]:
import jsonlines
# The data is now structured like this:
# [
#   {
#     'id': '1ABC',
#     'seq': 'MKT...',
#     'Phosphorylation': [
#        {"Site": 45, "LOGSUM": -12.34},
#        ...
#     ],
#     // remaining 4 PTMs in the same order as Phosphorylation
#   },
# ]

# Now to write this to JSON file
with jsonlines.open('pdb_ptm_scores.json', mode='w') as writer:
    for i, row in enumerate(processed_data):
        if i % 10000 == 0 and i != 0:
            print("Written", i, "entries to file")
        writer.write(row)
    print("Scores written to pdb_ptm_scores.json")

Written 10000 entries to file
Written 20000 entries to file
Written 30000 entries to file
Written 40000 entries to file
Written 50000 entries to file
Written 60000 entries to file
Written 70000 entries to file
Written 80000 entries to file
Written 90000 entries to file
Written 100000 entries to file
Written 110000 entries to file
Written 120000 entries to file
Written 130000 entries to file
Written 140000 entries to file
Written 150000 entries to file
Written 160000 entries to file
Written 170000 entries to file
Written 180000 entries to file
Written 190000 entries to file
Written 200000 entries to file
Written 210000 entries to file
Written 220000 entries to file
Written 230000 entries to file
Written 240000 entries to file
Written 250000 entries to file
Written 260000 entries to file
Written 270000 entries to file
Written 280000 entries to file
Written 290000 entries to file
Written 300000 entries to file
Written 310000 entries to file
Written 320000 entries to file
Written 330000 en

In [30]:
len(processed_data)

989835